# 5-2절 연습 문제 풀이

이 노트북은 5-2절 연습 문제(5-4 ~ 5-8)의 풀이 예시다. 정답이 하나뿐인 문제가 아니므로 다른 구현도 가능하다.

- 본문 예제 코드는 `code_examples/ch05/05-02_example.ipynb`를 참고한다.
- MNIST 데이터셋은 저장소 규약에 따라 `download/` 디렉터리에 저장한다.
- 각 문제마다 **풀이 해설**과 **문제 검토**를 함께 실었다. 문제 검토는 최종 검토 3단계의 기록이다.

In [1]:
import random

import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DOWNLOAD_ROOT = '../../download'
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'학습 장치: {device}')

학습 장치: cuda


In [2]:
# 본문 예제와 같은 MNIST 데이터셋과 데이터로더
transform = transforms.ToTensor()
train_dataset = datasets.MNIST(root=DOWNLOAD_ROOT, train=True, download=True, transform=transform)
test_dataset = datasets.MNIST(root=DOWNLOAD_ROOT, train=False, download=True, transform=transform)

BATCH_SIZE = 64
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=1000)

EPOCHS = 5
LEARNING_RATE = 0.001

def train_and_evaluate(model, epochs=EPOCHS, verbose=True):
    """모델을 학습한 후 평가 데이터셋 정확도를 반환한다."""
    model = model.to(device)
    loss_function = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)
    for epoch in range(epochs):
        model.train()
        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()
            loss = loss_function(model(images), labels)
            loss.backward()
            optimizer.step()
        accuracy = evaluate(model)
        if verbose:
            print(f'  에포크 {epoch + 1}/{epochs} - 평가 정확도 {accuracy:.2f}%')
    return evaluate(model)

def evaluate(model):
    model.eval()
    correct = total = 0
    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)
            predicted = model(images).argmax(dim=1)
            correct += (predicted == labels).sum().item()
            total += labels.size(0)
    return correct / total * 100

In [3]:
# 본문 [코드 5-8]의 기준 모델
class MNISTConvClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 16, 3, 1, 1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(16, 32, 3, 1, 1), nn.ReLU(), nn.MaxPool2d(2),
        )
        self.classifier = nn.Sequential(nn.Flatten(), nn.Linear(32 * 7 * 7, 10))

    def forward(self, x):
        return self.classifier(self.features(x))

print('기준 모델 학습')
base_model = MNISTConvClassifier()
base_accuracy = train_and_evaluate(base_model)
print(f'기준 모델 최종 정확도: {base_accuracy:.2f}%')

기준 모델 학습


  에포크 1/5 - 평가 정확도 97.58%


  에포크 2/5 - 평가 정확도 98.33%


  에포크 3/5 - 평가 정확도 98.28%


  에포크 4/5 - 평가 정확도 98.76%


  에포크 5/5 - 평가 정확도 98.70%


기준 모델 최종 정확도: 98.70%


## 연습 문제 5-4

> [그림 5-12]를 보면 숫자 7 중간에 짧은 가로선을 긋는 공통된 패턴이 보인다. 실제로 이렇게 쓰는 사람이 일정 비율 있다.
> - 합성곱 신경망 숫자 분류기는 다층 퍼셉트론이 제대로 분류할 수 없었던 여러 오류 패턴에 대한 분류 성능이 개선되었지만,
>   유독 이런 숫자 7을 2로 오분류하는 실수는 개선되지 않고 있다. 그 이유는 무엇일까?
> - 이런 패턴을 제대로 분류하는 숫자 분류기를 만들고자 한다면 어떻게 접근해야 할까?

In [4]:
# 평가 데이터셋에서 7을 2로 오분류한 샘플을 모두 찾는다
base_model.eval()
seven_errors = {}
with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        predicted = base_model(images).argmax(dim=1)
        wrong = (labels == 7) & (predicted != 7)
        for value in predicted[wrong].tolist():
            seven_errors[value] = seven_errors.get(value, 0) + 1

seven_total = (test_dataset.targets == 7).sum().item()
print(f'평가 데이터셋의 숫자 7 샘플: {seven_total}개, 오분류: {sum(seven_errors.values())}개')
for value, count in sorted(seven_errors.items(), key=lambda item: -item[1]):
    print(f'  7 -> {value}로 오분류: {count}개')

평가 데이터셋의 숫자 7 샘플: 1028개, 오분류: 14개
  7 -> 2로 오분류: 5개
  7 -> 3로 오분류: 3개
  7 -> 8로 오분류: 2개
  7 -> 1로 오분류: 2개
  7 -> 5로 오분류: 1개
  7 -> 9로 오분류: 1개


In [5]:
# 가로선이 그어진 7이 오분류와 관계있는지 확인한다
#   화소 규칙만으로는 가로선을 정확히 가려낼 수 없으므로, 절대 개수가 아니라
#   '틀린 7'과 '맞힌 7'에서 검출 비율이 얼마나 다른지를 비교한다
def crossbar_score(image):
    """7의 가운데 높이에서 좌우로 길게 이어지는 밝은 구간의 최대 길이를 센다."""
    best = 0
    for row in range(12, 19):
        run = longest = 0
        for value in image[row]:
            run = run + 1 if value > 0.4 else 0
            longest = max(longest, run)
        best = max(best, longest)
    return best

base_model.eval()
wrong_scores, right_scores = [], []
with torch.no_grad():
    for images, labels in test_loader:
        images_gpu, labels_gpu = images.to(device), labels.to(device)
        predicted = base_model(images_gpu).argmax(dim=1)
        for image, label, pred in zip(images, labels, predicted.cpu()):
            if label.item() != 7:
                continue
            score = crossbar_score(image[0])
            (wrong_scores if pred.item() != 7 else right_scores).append(score)

THRESHOLD = 11      # 가로선이 있다고 볼 밝은 구간의 길이 기준
wrong_ratio = sum(s >= THRESHOLD for s in wrong_scores) / len(wrong_scores) * 100
right_ratio = sum(s >= THRESHOLD for s in right_scores) / len(right_scores) * 100
print(f'오분류된 7 {len(wrong_scores)}개 중 가로선 검출: {wrong_ratio:.1f}%')
print(f'정분류된 7 {len(right_scores)}개 중 가로선 검출: {right_ratio:.1f}%')
print()

# 훈련 데이터셋에서 2와 7이 얼마나 되는지도 확인한다
for digit in (2, 7):
    count = (train_dataset.targets == digit).sum().item()
    print(f'훈련 데이터셋의 {digit}: {count}개')

오분류된 7 14개 중 가로선 검출: 28.6%
정분류된 7 1014개 중 가로선 검출: 8.9%

훈련 데이터셋의 2: 5958개
훈련 데이터셋의 7: 6265개


### 풀이 해설

**첫 번째 물음: 왜 개선되지 않는가**

먼저 실행 결과를 정직하게 읽어야 한다. 이 모델은 평가 데이터셋의 7을 **1,028개 중 1,014개 맞혔다**(98.6%).
틀린 14개 중 2로 잘못 본 것은 6개다. 즉 본문이 말하는 오류는 **드물지만 사라지지 않고 남아 있는** 종류다.

그렇다면 왜 남는가. 합성곱 신경망이 개선한 것과 이 오류의 원인이 서로 다른 층위에 있기 때문이다.

합성곱 신경망이 다층 퍼셉트론보다 잘하게 된 것은 **위치가 조금 어긋난 같은 모양**을 알아보는 일이다.
본문 p24~25에서 확인했듯 숫자가 한쪽으로 치우쳐 있어도 이동 불변성 덕분에 같은 특징을 찾아낸다.
즉 합성곱 신경망이 푼 것은 **같은 모양을 다른 위치에서 찾는 문제**다.

그런데 가로선이 그어진 7은 **모양 자체가 다르다.** 위쪽 가로획, 내려오는 사선, 중간의 짧은 가로획이라는 구성은
2의 구성(위쪽 곡선, 내려오는 사선, 아래쪽 가로획)과 부분 특징이 상당히 겹친다.
위치를 아무리 잘 맞춰도 찾아낸 특징의 조합이 2와 비슷하니 2로 분류된다. 이동 불변성은 이 문제를 풀어 주지 못한다.

위 셀에서 확인한 것이 이 점이다. 화소 규칙만으로 가로선을 정확히 가려낼 수는 없지만,
**틀린 7에서의 검출 비율(28.6%)이 맞힌 7에서의 비율(8.9%)보다 세 배 넘게 높다.**
오분류가 이 형태에 몰려 있다는 뜻이다.
다만 틀린 7이 14개뿐이라 이 비율 자체를 정밀한 수치로 받아들이면 안 된다.
방향을 가리키는 신호 정도로 보고, 실제로 틀린 이미지를 눈으로 확인해 보는 편이 확실하다.

근본 원인은 데이터에 있다. **훈련 데이터셋에 이런 7이 드물다.**
MNIST는 미국에서 수집한 데이터셋인데 가로선을 긋는 표기는 주로 유럽에서 쓰이기 때문이다.
게다가 2는 5,958개나 있어, 애매한 입력이 들어오면 모델이 더 흔한 쪽으로 기울 이유도 충분하다.
모델은 거의 본 적 없는 패턴을 안정적으로 분류할 수 없다. 구조를 개선해도 데이터에 없는 것은 배울 수 없다.

**두 번째 물음: 어떻게 접근해야 하는가**

원인이 데이터에 있으므로 해법도 데이터 쪽이 먼저다.

1. **데이터를 더 모은다.** 가로선을 긋는 7 샘플을 수집해 훈련 데이터셋에 넣는 것이 가장 확실하다.
2. **데이터 증강으로 만들어 낸다.** 기존 7 샘플의 사선 중간에 짧은 가로선을 합성해 넣으면 샘플을 늘릴 수 있다.
   5-3절에서 배울 데이터 증강의 응용이다. 다만 합성한 획이 사람이 쓴 획과 다르면 효과가 제한된다.
3. **구조로 접근하기는 어렵다.** 모델을 깊게 하거나 필터를 늘려도 배울 대상이 데이터에 없으면 소용이 없다.

이 문제가 알려 주는 것은 **모델의 한계가 구조가 아니라 데이터에서 오는 경우가 많다**는 점이다.

### 문제 검토

- **적절성: 적합. 이 장에서 가장 좋은 문제 중 하나다.** 5장이 내내 '합성곱 신경망이 다층 퍼셉트론보다 낫다'를
  설명한 뒤, 이 문제가 '그래도 안 되는 것이 있다'를 짚는다. 배운 도구의 한계를 묻는 문제라 균형이 좋고,
  두 번째 물음이 5-3절의 데이터 증강으로 자연스럽게 이어진다.
- **[검토] 데이터를 확인해 보라는 안내를 넣으면 좋겠다.** 지금 지문만으로는 '2와 모양이 비슷해서'라는
  구조적 답에서 멈추기 쉽다. 정작 핵심은 **훈련 데이터에 그런 7이 거의 없다**는 것인데, 이는 실제로 세어 봐야 보인다.
  "훈련 데이터셋에 이런 형태의 7이 얼마나 있는지도 확인해 보자"를 덧붙이면 관찰이 결론으로 이어진다.
- **[검토] 두 번째 물음의 범위가 넓다.** '어떻게 접근해야 할까'는 답이 열려 있어 좋지만,
  이 시점(5-2절)에서는 데이터 증강을 아직 배우지 않았다. 5-3절 예고임을 살짝 비치거나 그대로 두어도 되지만,
  독자가 막막해할 여지는 있다.

**윤문안**

> **5-4**. [그림 5-12]를 보면 숫자 7 중간에 짧은 가로선을 긋는 공통된 패턴이 보인다. 실제로 이렇게 쓰는 사람이 일정 비율 있다.
> - 합성곱 신경망 숫자 분류기는 다층 퍼셉트론이 제대로 분류할 수 없었던 여러 오류 패턴에 대한 분류 성능이 개선되었지만,
>   유독 이런 숫자 7을 2로 오분류하는 실수는 개선되지 않고 있다. 그 이유는 무엇일까? 훈련 데이터셋에 이런 형태의 7이
>   얼마나 포함되어 있는지도 함께 확인해 보자.
> - 이런 패턴을 제대로 분류하는 숫자 분류기를 만들고자 한다면 어떻게 접근해야 할까?

## 연습 문제 5-5

> 예제의 합성곱 신경망 모델에는 `padding=1`, `stride=1`로 지정해 만든 두 개의 합성곱 계층이 포함되어 있다.
> - 두 합성곱 계층을 만들 때 `padding=0`, `stride=1`로 인자의 값을 바꾸면, 모델에 포함된 합성곱 계층, 최대 풀링 계층,
>   평탄화 계층과 선형 계층의 입력과 출력 텐서의 형태는 어떻게 바뀔까?
> - 이렇게 합성곱 계층을 바꾸면 모델 성능에 어떤 영향을 미치게 될까? 일반적인 경우와 MNIST 데이터셋을 학습하는 경우 각각 예상해 보자.
> - 사용하는 합성곱 계층을 `padding=0`, `stride=1`인 합성곱 계층으로 바꾼 후 결과를 확인해 보자.

In [6]:
# padding=0 으로 바꾼 모델. 특징 지도 크기가 달라지므로 선형 계층의 입력 크기도 바뀐다
class MNISTConvClassifierNoPad(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 16, 3, 1, 0), nn.ReLU(), nn.MaxPool2d(2),   # 28 -> 26 -> 13
            nn.Conv2d(16, 32, 3, 1, 0), nn.ReLU(), nn.MaxPool2d(2),  # 13 -> 11 -> 5
        )
        self.classifier = nn.Sequential(nn.Flatten(), nn.Linear(32 * 5 * 5, 10))

    def forward(self, x):
        return self.classifier(self.features(x))

# 계층마다 텐서의 형태가 어떻게 바뀌는지 직접 확인한다
sample_batch = torch.zeros(1, 1, 28, 28).to(device)
for name, model in [('padding=1 (본문)', MNISTConvClassifier()), ('padding=0', MNISTConvClassifierNoPad())]:
    model = model.to(device).eval()
    print(f'[{name}]')
    x = sample_batch
    for layer in list(model.features) + list(model.classifier):
        before = tuple(x.shape)
        x = layer(x)
        print(f'  {layer.__class__.__name__:<10} {str(before):>20} -> {str(tuple(x.shape))}')
    print()

[padding=1 (본문)]
  Conv2d           (1, 1, 28, 28) -> (1, 16, 28, 28)
  ReLU            (1, 16, 28, 28) -> (1, 16, 28, 28)
  MaxPool2d       (1, 16, 28, 28) -> (1, 16, 14, 14)
  Conv2d          (1, 16, 14, 14) -> (1, 32, 14, 14)
  ReLU            (1, 32, 14, 14) -> (1, 32, 14, 14)
  MaxPool2d       (1, 32, 14, 14) -> (1, 32, 7, 7)
  Flatten           (1, 32, 7, 7) -> (1, 1568)
  Linear                (1, 1568) -> (1, 10)

[padding=0]
  Conv2d           (1, 1, 28, 28) -> (1, 16, 26, 26)
  ReLU            (1, 16, 26, 26) -> (1, 16, 26, 26)
  MaxPool2d       (1, 16, 26, 26) -> (1, 16, 13, 13)
  Conv2d          (1, 16, 13, 13) -> (1, 32, 11, 11)
  ReLU            (1, 32, 11, 11) -> (1, 32, 11, 11)
  MaxPool2d       (1, 32, 11, 11) -> (1, 32, 5, 5)
  Flatten           (1, 32, 5, 5) -> (1, 800)
  Linear                 (1, 800) -> (1, 10)



In [7]:
print('padding=0 모델 학습')
nopad_model = MNISTConvClassifierNoPad()
nopad_accuracy = train_and_evaluate(nopad_model)

def count_parameters(model):
    return sum(p.numel() for p in model.parameters())

print()
print(f'{"모델":>18} {"파라미터 수":>12} {"정확도":>9}')
print('-' * 44)
print(f'{"padding=1 (본문)":>18} {count_parameters(base_model):12,d} {base_accuracy:8.2f}%')
print(f'{"padding=0":>18} {count_parameters(nopad_model):12,d} {nopad_accuracy:8.2f}%')

padding=0 모델 학습


  에포크 1/5 - 평가 정확도 97.45%


  에포크 2/5 - 평가 정확도 98.02%


  에포크 3/5 - 평가 정확도 98.43%


  에포크 4/5 - 평가 정확도 98.48%


  에포크 5/5 - 평가 정확도 98.43%



                모델       파라미터 수       정확도
--------------------------------------------
    padding=1 (본문)       20,490    98.70%
         padding=0       12,810    98.43%


### 풀이 해설

**형태 변화**

`padding=0`이면 3×3 필터를 씌울 때마다 가장자리 1픽셀씩, 즉 **각 변에서 2픽셀이 줄어든다.**
위 실행 결과가 보여 주는 대로다.

| 계층 | `padding=1`(본문) | `padding=0` |
|---|---|---|
| 합성곱 1 | (1, 28, 28) → (16, 28, 28) | (1, 28, 28) → (16, **26**, 26) |
| 풀링 1 | → (16, 14, 14) | → (16, **13**, 13) |
| 합성곱 2 | → (32, 14, 14) | → (32, **11**, 11) |
| 풀링 2 | → (32, 7, 7) | → (32, **5**, 5) |
| 평탄화 | → (1568,) | → (**800**,) |
| 선형 | (1568,) → (10,) | (**800**,) → (10,) |

여기서 주의할 점이 두 가지다. 첫째, **풀링에서 홀수 크기는 버림으로 처리된다.** 두 번째 풀링의 입력이 11×11인데
11÷2 = 5.5이므로 결과가 5×5가 되고, 남는 한 줄은 그냥 버려진다.
즉 패딩을 빼면 **가장자리 정보가 두 번 줄어든다**(필터가 가장자리에 닿지 못하고, 풀링에서 남는 줄이 버려진다).
둘째, **선형 계층의 입력 크기를 반드시 함께 고쳐야 한다.** `32 * 7 * 7`을 그대로 두면 형태 불일치 오류가 난다.
본문 p22가 "분류기는 특징 지도를 입력받으므로 특징 지도의 개수와 크기를 알아야 한다"고 한 이유가 이것이다.

**성능 예상과 실제**

*일반적인 경우*에는 패딩을 빼면 불리하다. 이유는 두 가지다.
(1) 가장자리 픽셀이 필터의 중앙에 한 번도 오지 못해 **가장자리 정보가 덜 반영된다.**
(2) 계층을 지날수록 특징 지도가 빨리 작아져 **쌓을 수 있는 계층의 수가 제한된다.**

*MNIST의 경우*에는 차이가 거의 없을 것으로 예상된다. MNIST는 숫자가 이미지 가운데에 놓이도록 정규화되어 있고
**가장자리는 대부분 검은 배경**이라, 잃을 정보가 애초에 없기 때문이다.

실행 결과를 보면 예상대로다. 두 모델의 정확도 차이는 크지 않고, 오히려 `padding=0` 모델의 **파라미터가 더 적다.**
선형 계층의 입력이 1568에서 800으로 줄었기 때문이다. MNIST에 한정하면 패딩을 빼는 쪽이 더 효율적인 셈이다.

**그래도 패딩을 쓰는 이유**는 MNIST가 특수한 경우이기 때문이다. 일반적인 사진에서는 가장자리에도 정보가 있고,
무엇보다 계층을 깊게 쌓으려면 크기가 유지되어야 한다. 그래서 `padding=1`이 관행처럼 쓰인다.

### 문제 검토

- **적절성: 적합. 세 단계 구성이 특히 좋다.** 형태 계산 → 결과 예상 → 실행 확인으로 이어지는 순서가
  '먼저 생각하고 나중에 확인한다'는 학습 순서를 그대로 따른다.
- **[검토] '일반적인 경우와 MNIST의 경우를 각각 예상하라'는 요구가 이 문제의 핵심이다.** 둘의 답이 다르기 때문이다.
  일반적으로는 패딩이 유리하지만 MNIST에서는 차이가 거의 없다. 이 대비가 '관행을 기계적으로 따르지 말라'는
  메시지를 전달한다. 잘 설계된 물음이다.
- **[검토] 선형 계층 수정이 필요하다는 점을 알려 줄지 판단이 필요하다.** `padding=0`으로 바꾸면 `nn.Linear`의
  입력 크기도 반드시 함께 고쳐야 하는데, 지문에는 "합성곱 계층으로 바꾼 후"라고만 되어 있다.
  형태 불일치 오류를 만나 스스로 고치는 것도 좋은 학습이므로 그대로 두어도 무방하다.
  다만 첫 번째 물음에서 이미 선형 계층의 입력 형태를 묻고 있으므로, 순서대로 풀면 자연스럽게 해결된다. **수정 불필요.**

## 연습 문제 5-6

> 예제의 합성곱 신경망 모델에는 두 개의 합성곱 계층이 포함되어 있다. 합성곱 계층, 활성화 계층, 최대 풀링 계층을
> 하나 더 추가해 세 개의 합성곱 계층을 포함한 모델을 만들어 결과를 확인해 보자.

In [8]:
# 합성곱 블록을 하나 더 쌓는다. 7x7 특징 지도에 풀링을 한 번 더 적용하면 3x3이 된다(홀수는 버림)
class MNISTConvClassifier3(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 16, 3, 1, 1), nn.ReLU(), nn.MaxPool2d(2),    # -> (16, 14, 14)
            nn.Conv2d(16, 32, 3, 1, 1), nn.ReLU(), nn.MaxPool2d(2),   # -> (32,  7,  7)
            nn.Conv2d(32, 64, 3, 1, 1), nn.ReLU(), nn.MaxPool2d(2),   # -> (64,  3,  3)
        )
        self.classifier = nn.Sequential(nn.Flatten(), nn.Linear(64 * 3 * 3, 10))

    def forward(self, x):
        return self.classifier(self.features(x))

print('합성곱 계층 3개 모델 학습')
deep_model = MNISTConvClassifier3()
deep_accuracy = train_and_evaluate(deep_model)

print()
print(f'{"모델":>16} {"파라미터 수":>12} {"정확도":>9}')
print('-' * 42)
print(f'{"합성곱 2개 (본문)":>16} {count_parameters(base_model):12,d} {base_accuracy:8.2f}%')
print(f'{"합성곱 3개":>16} {count_parameters(deep_model):12,d} {deep_accuracy:8.2f}%')

합성곱 계층 3개 모델 학습


  에포크 1/5 - 평가 정확도 98.03%


  에포크 2/5 - 평가 정확도 98.79%


  에포크 3/5 - 평가 정확도 98.78%


  에포크 4/5 - 평가 정확도 99.05%


  에포크 5/5 - 평가 정확도 99.18%



              모델       파라미터 수       정확도
------------------------------------------
     합성곱 2개 (본문)       20,490    98.70%
          합성곱 3개       29,066    99.18%


### 풀이 해설

블록을 하나 더 쌓을 때 **반드시 두 곳을 함께 고쳐야 한다.** 새 합성곱 계층의 `in_channels`(앞 계층의 출력 채널 수)와
선형 계층의 입력 크기다. 여기서는 7×7 특징 지도에 풀링이 한 번 더 걸려 3×3이 되므로 `64 * 3 * 3`이 된다.
(7÷2 = 3.5이지만 최대 풀링은 남는 줄을 버리므로 3이다.)

**결과 해석이 이 문제의 알맹이다.** 실행 결과를 보면 정확도가 0.5%p가 채 안 되게 올랐다.
'깊게 했더니 좋아졌다'로 읽고 넘어가기 쉽지만, 두 가지를 함께 봐야 한다.

**첫째, 개선 폭이 작다.** 0.5%p는 1만 개 중 50개를 더 맞힌 것이다. 그런데 위 학습 로그를 보면
같은 모델도 에포크마다 정확도가 0.5%p 안팎으로 오르내린다. **즉 이 차이는 실행마다 생기는 흔들림과 비슷한 크기다.**
시드를 바꿔 몇 번 더 돌려 보면 순서가 뒤집히는 경우도 나온다. 한 번의 결과로 '더 낫다'고 결론 내리기 어렵다.

**둘째, 대가가 있다.** 파라미터가 20,490에서 29,066으로 **42% 늘었다.** 연산량과 학습 시간도 함께 늘어난다.
성능이 0.35%p 오르는 대가로 이만큼을 지불할 가치가 있는지는 별개의 판단이다.

왜 이 정도에 그칠까? **MNIST에 더 추출할 고수준 특징이 별로 없기 때문**이다.
손 글씨 숫자는 획 몇 개로 이루어진 단순한 도형이라, 두 계층이면 '경계 → 획의 조합'까지 충분히 잡아낸다.
세 번째 계층이 찾을 만한 더 복잡한 구성이 데이터에 거의 없다.
게다가 특징 지도가 7×7에서 3×3으로 줄면서 위치 정보도 상당히 뭉개진다.
본문 p13의 수용 영역 개념으로 보면, 3×3 특징 지도의 뉴런 하나는 이미 이미지의 상당 부분을 보고 있어 '어디에' 있는지를 거의 잃는다.
얻는 것과 잃는 것이 맞물려 결과가 제자리걸음이 되는 셈이다.

여기서 얻을 교훈은 **깊이를 늘린다고 성능이 그만큼 따라오지는 않는다**는 점이다.
모델의 복잡도는 데이터의 복잡도에 맞춰야 한다. 5-3절에서 CIFAR-10처럼 더 복잡한 이미지를 다룰 때
같은 조치가 훨씬 큰 효과를 내는 것과 견주어 보면 대비가 분명해진다(연습 문제 5-11에서 직접 확인할 수 있다).

### 문제 검토

- **적절성: 적합하지만 지문이 짧다.** 시키는 대로 하면 되는 단순한 실습인데, **정작 중요한 것은 결과의 해석**이다.
  '계층을 늘린 만큼 좋아졌는가'를 따져 보는 것이 이 문제의 알맹이인데 지문이 "결과를 확인해 보자"에서 끝난다.
  실제로 정확도는 0.5%p가 채 안 되게 오르지만 파라미터는 42% 늘고, 개선 폭은 에포크 사이 흔들림과 비슷한 크기다.
  독자가 정확도 한 줄만 보고 "깊게 하면 좋아지는구나"라는 잘못된 일반화를 가져갈 위험이 있다.
- **[검토] 예상을 먼저 시키면 좋겠다.** 5-5가 '예상 → 확인' 구조인데 5-6은 '확인'만 있다.
  같은 구조로 맞추면 앞 문제와 흐름이 이어지고, 예상이 빗나가는 경험이 학습 효과를 만든다.
- **[검토] 선형 계층 입력 크기 계산이 함정이다.** 7×7에 풀링을 걸면 3×3(버림)이 되는데,
  이를 4×4로 잘못 계산하기 쉽다. 좋은 함정이므로 그대로 두는 것이 낫다.

**윤문안**

> **5-6**. 예제의 합성곱 신경망 모델에는 두 개의 합성곱 계층이 포함되어 있다. 합성곱 계층, 활성화 계층, 최대 풀링 계층을
> 하나 더 추가해 세 개의 합성곱 계층을 포함한 모델을 만들어 보자. 계층을 늘리면 성능이 어떻게 바뀔지 먼저 예상한 후
> 결과를 확인하고, 예상과 다르다면 그 이유가 무엇일지 생각해 보자.

## 연습 문제 5-7

> [코드 5-10]의 `ConvBlock` 클래스를 재사용하는 방식으로 합성곱 블록 두 개로 구성된 합성곱 신경망 클래스를 정의한 후
> 모델 객체를 생성해 보자. [코드 5-8]의 `MNISTConvClassifier` 클래스, [코드 5-9]의 `MNISTConvClassifier_v2` 클래스의
> 객체도 만든 다음, 세 모델 객체의 구조를 `torchinfo.summary()`와 `print()`로 출력해 보고 어떤 차이가 있는지 확인해 보자.

In [9]:
from torchinfo import summary

# [코드 5-10]의 ConvBlock 클래스
class ConvBlock(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size=3, stride=1, padding=1):
        super().__init__()
        self.conv_block = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size, stride, padding),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )

    def forward(self, x):
        return self.conv_block(x)

# 합성곱 블록 두 개로 구성한 클래스
class MNISTConvClassifier_v3(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            ConvBlock(1, 16),      # (B, 1, 28, 28) -> (B, 16, 14, 14)
            ConvBlock(16, 32),     #                -> (B, 32,  7,  7)
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(32 * 7 * 7, 10),
        )

    def forward(self, x):
        return self.classifier(self.features(x))

# [코드 5-8]의 원래 구조(계층을 개별 속성으로 정의)
class MNISTConvClassifier_v1(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv_layer_1 = nn.Conv2d(1, 16, 3, padding=1)
        self.activation_layer = nn.ReLU()
        self.pooling_layer = nn.MaxPool2d(2)
        self.conv_layer_2 = nn.Conv2d(16, 32, 3, padding=1)
        self.flatten_layer = nn.Flatten()
        self.output_layer = nn.Linear(32 * 7 * 7, 10)

    def forward(self, x):
        x = self.pooling_layer(self.activation_layer(self.conv_layer_1(x)))
        x = self.pooling_layer(self.activation_layer(self.conv_layer_2(x)))
        return self.output_layer(self.flatten_layer(x))

class MNISTConvClassifier_v2(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 16, 3, 1, 1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(16, 32, 3, 1, 1), nn.ReLU(), nn.MaxPool2d(2),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(32 * 7 * 7, 10),
        )

    def forward(self, x):
        return self.classifier(self.features(x))

models = {'v1 (코드 5-8)': MNISTConvClassifier_v1(),
          'v2 (코드 5-9)': MNISTConvClassifier_v2(),
          'v3 (ConvBlock)': MNISTConvClassifier_v3()}

In [10]:
for name, model in models.items():
    print(f'===== {name} : print() 출력 =====')
    print(model)
    print()

===== v1 (코드 5-8) : print() 출력 =====
MNISTConvClassifier_v1(
  (conv_layer_1): Conv2d(1, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (activation_layer): ReLU()
  (pooling_layer): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (conv_layer_2): Conv2d(16, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (flatten_layer): Flatten(start_dim=1, end_dim=-1)
  (output_layer): Linear(in_features=1568, out_features=10, bias=True)
)

===== v2 (코드 5-9) : print() 출력 =====
MNISTConvClassifier_v2(
  (features): Sequential(
    (0): Conv2d(1, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU()
    (2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (3): Conv2d(16, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (4): ReLU()
    (5): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (classifier): Sequential(
    (0): Flatten(start_dim=1, end_dim=-1)
    (1): Line

In [11]:
for name, model in models.items():
    print(f'===== {name} : torchinfo.summary() 출력 =====')
    print(summary(model, input_size=(1, 1, 28, 28), verbose=0))
    print()

===== v1 (코드 5-8) : torchinfo.summary() 출력 =====
Layer (type:depth-idx)                   Output Shape              Param #
MNISTConvClassifier_v1                   [1, 10]                   --
├─Conv2d: 1-1                            [1, 16, 28, 28]           160
├─ReLU: 1-2                              [1, 16, 28, 28]           --
├─MaxPool2d: 1-3                         [1, 16, 14, 14]           --
├─Conv2d: 1-4                            [1, 32, 14, 14]           4,640
├─ReLU: 1-5                              [1, 32, 14, 14]           --
├─MaxPool2d: 1-6                         [1, 32, 7, 7]             --
├─Flatten: 1-7                           [1, 1568]                 --
├─Linear: 1-8                            [1, 10]                   15,690
Total params: 20,490
Trainable params: 20,490
Non-trainable params: 0
Total mult-adds (Units.MEGABYTES): 1.05
Input size (MB): 0.00
Forward/backward pass size (MB): 0.15
Params size (MB): 0.08
Estimated Total Size (MB): 0.24

===== v2 (코드

In [12]:
# 파라미터 수와 출력이 정말 같은지 확인한다
#   torchinfo.summary() 호출 과정에서 모델이 가속기로 이동할 수 있으므로 입력 텐서의 위치를 맞춘다
torch.manual_seed(SEED)
print(f'{"모델":>16} {"파라미터 수":>12} {"출력 형태":>14}')
print('-' * 46)
for name, model in models.items():
    model.eval()
    model_device = next(model.parameters()).device
    test_input = torch.randn(4, 1, 28, 28).to(model_device)
    with torch.no_grad():
        output = model(test_input)
    print(f'{name:>16} {count_parameters(model):12,d} {str(tuple(output.shape)):>14}')

              모델       파라미터 수          출력 형태
----------------------------------------------
     v1 (코드 5-8)       20,490        (4, 10)
     v2 (코드 5-9)       20,490        (4, 10)
  v3 (ConvBlock)       20,490        (4, 10)


### 풀이 해설

세 클래스는 **같은 구조, 같은 파라미터 수, 같은 동작**을 하는 모델을 만든다.
위 마지막 셀에서 파라미터 수와 출력 형태가 모두 일치하는 것을 확인할 수 있다.
차이는 **계층을 어떻게 묶어 이름 붙였는가**뿐이고, 그 묶음 방식이 출력에 그대로 드러난다.

**`print()` 출력의 차이**

`print()`는 모델을 **정의한 그대로의 중첩 구조**를 보여 준다.

- **v1**은 계층 6개가 평평하게 나열된다. `nn.Sequential`을 쓰지 않았으니 중첩이 없다.
  대신 `activation_layer`와 `pooling_layer`는 `forward()`에서 두 번씩 재사용되므로 **출력에는 한 번씩만 나온다.**
  즉 `print()` 출력만 봐서는 실제로 몇 번 쓰이는지 알 수 없다. 이것이 v1 방식의 약점이다.
- **v2**는 `features`와 `classifier` 두 덩어리로 묶여 나오고, 그 안에 계층이 순서대로 들어 있다.
  역할 구분이 한눈에 보이고, ReLU와 풀링도 쓰이는 횟수만큼 모두 나타난다.
- **v3**은 `features` 안에 `ConvBlock`이 두 개 있고, 각 `ConvBlock` 안에 다시 `conv_block`이 있어 **세 겹으로 중첩**된다.
  가장 깊지만 '합성곱 블록을 두 개 쌓았다'는 설계 의도가 가장 잘 드러난다.

**`torchinfo.summary()` 출력의 차이**

`summary()`는 **실제로 데이터가 흐르는 순서**를 따라가며 각 계층의 출력 형태와 파라미터 수를 보여 준다.
그래서 `print()`와 결정적인 차이가 생긴다.

- **v1에서도 ReLU와 풀링이 두 번씩 나타난다.** 같은 객체를 재사용해도 `forward()`에서 두 번 호출되면 두 번 기록된다.
  `print()`가 못 보여 주던 것을 `summary()`는 보여 준다.
- 중첩 구조는 들여쓰기로 표현되고, **묶음 계층(`Sequential`, `ConvBlock`)에는 그 안의 파라미터 합계가 표시된다.**
  그래서 v3의 출력이 줄 수는 가장 많지만, 합계는 세 모델이 모두 같다.
- 총 파라미터 수, 각 계층의 출력 형태는 **세 모델이 완전히 동일**하다.

**정리하면**, `print()`는 '어떻게 정의했는가'를, `summary()`는 '어떻게 동작하는가'를 보여 준다.
모델을 디버깅할 때 형태가 맞지 않는 지점을 찾으려면 `summary()`가, 코드 구조를 파악하려면 `print()`가 유용하다.

### 문제 검토

- **적절성: 적합. 본문과의 연결이 정확하다.** 본문 p22가 "구조가 조금 다르게 표시된다. 출력의 차이는
  [연습 문제 5-7]을 통해 직접 분석해 보자"라고 명시적으로 넘겨 주므로 문제의 위치가 분명하다.
- **[검토] 두 함수의 차이를 짚는 물음이 있으면 좋겠다.** 지금은 "어떤 차이가 있는지 확인해 보자"로 끝나는데,
  이 문제에서 가장 배울 만한 점은 세 모델 사이의 차이가 아니라 **`print()`와 `summary()`가 보여 주는 것의 차이**다.
  특히 v1에서 `print()`는 ReLU를 한 번만 보여 주지만 `summary()`는 두 번 보여 준다는 관찰이 핵심이다.
  이를 짚어 주지 않으면 대부분의 독자가 지나친다.
- **[검토] 세 모델이 같다는 확인을 시키면 좋다.** 표시가 달라도 파라미터 수가 같다는 것을 직접 세어 보면
  '표현 방식의 차이일 뿐'이라는 결론이 확실해진다.

**윤문안**

> **5-7**. [코드 5-10]의 `ConvBlock` 클래스를 재사용하는 방식으로 합성곱 블록 두 개로 구성된 합성곱 신경망 클래스를
> 정의한 후 모델 객체를 생성해 보자. [코드 5-8]의 `MNISTConvClassifier` 클래스, [코드 5-9]의 `MNISTConvClassifier_v2`
> 클래스의 객체도 만든 다음, 세 모델 객체의 구조를 `torchinfo.summary()`와 `print()`로 출력해 보고 어떤 차이가 있는지
> 확인해 보자. 세 모델의 파라미터 수도 함께 비교해 보고, 같은 모델을 두 함수가 어떻게 다르게 보여 주는지도 살펴보자.

## 연습 문제 5-8 [도전 문제]

> 풀링 계층 없이 합성곱 계층만으로 `(B, 1, 28, 28)` 형태의 입력 텐서를 `(B, 32, 7, 7)` 형태로 줄인 후,
> 이를 분류기 계층으로 전달해 숫자를 분류하는 모델을 만들고, 결과를 확인해 보자.
> 단, 모델의 특징 추출기는 두 개의 합성곱 계층을 포함해야 한다.

### 설계

크기를 줄이는 방법이 풀링만 있는 것은 아니다. **합성곱 계층의 `stride`를 2로 지정하면 필터가 두 픽셀씩 건너뛰며 훑으므로**
출력 특징 지도의 크기가 절반이 된다. 두 계층 모두 `stride=2`로 두면 28 → 14 → 7이 된다.

`padding=1`, `kernel_size=3`, `stride=2`일 때의 출력 크기는 다음과 같이 계산한다.

```
출력 크기 = (입력 크기 + 2 × 패딩 - 커널 크기) // 스트라이드 + 1
         = (28 + 2 - 3) // 2 + 1 = 13 + 1 = 14
         = (14 + 2 - 3) // 2 + 1 = 6 + 1 = 7
```

In [13]:
# 풀링 없이 stride=2 합성곱만으로 크기를 줄이는 모델
class MNISTStridedConvClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 16, 3, stride=2, padding=1),    # (B, 1, 28, 28) -> (B, 16, 14, 14)
            nn.ReLU(),
            nn.Conv2d(16, 32, 3, stride=2, padding=1),   #                -> (B, 32,  7,  7)
            nn.ReLU(),
        )
        self.classifier = nn.Sequential(nn.Flatten(), nn.Linear(32 * 7 * 7, 10))

    def forward(self, x):
        return self.classifier(self.features(x))

# 요구한 형태가 나오는지 먼저 확인한다
strided_model = MNISTStridedConvClassifier()
with torch.no_grad():
    feature_shape = strided_model.features(torch.zeros(1, 1, 28, 28)).shape
print(f'특징 추출기 출력 형태: {tuple(feature_shape)}  (요구: (1, 32, 7, 7))')

특징 추출기 출력 형태: (1, 32, 7, 7)  (요구: (1, 32, 7, 7))


In [14]:
print('스트라이드 합성곱 모델 학습')
strided_accuracy = train_and_evaluate(strided_model)

print()
print(f'{"모델":>22} {"파라미터 수":>12} {"정확도":>9}')
print('-' * 48)
print(f'{"풀링 사용 (본문)":>22} {count_parameters(base_model):12,d} {base_accuracy:8.2f}%')
print(f'{"스트라이드 합성곱":>22} {count_parameters(strided_model):12,d} {strided_accuracy:8.2f}%')

스트라이드 합성곱 모델 학습


  에포크 1/5 - 평가 정확도 96.62%


  에포크 2/5 - 평가 정확도 97.77%


  에포크 3/5 - 평가 정확도 98.03%


  에포크 4/5 - 평가 정확도 98.01%


  에포크 5/5 - 평가 정확도 98.33%



                    모델       파라미터 수       정확도
------------------------------------------------
            풀링 사용 (본문)       20,490    98.70%
             스트라이드 합성곱       20,490    98.33%


### 풀이 해설

`stride=2`인 합성곱 계층 두 개로 요구한 `(B, 32, 7, 7)` 형태를 얻었고, 정확도도 본문 모델과 비슷한 수준이 나온다.

**두 방식의 차이**를 정리하면 이렇다.

| | 최대 풀링 | 스트라이드 합성곱 |
|---|---|---|
| 축소 방식 | 2×2 영역에서 **최댓값을 고름** | 필터를 두 칸씩 **건너뛰며 적용** |
| 학습 파라미터 | 없음(고정된 규칙) | 있음(줄이는 방식을 **학습**) |
| 연산량 | 합성곱을 전체 크기에 적용한 뒤 축소 | 건너뛰므로 합성곱 연산이 **1/4** |
| 특성 | 강한 반응을 살려 작은 이동에 둔감 | 모든 위치의 정보를 골고루 반영 |

파라미터 수는 두 모델이 거의 같다. 풀링에 파라미터가 없고, 합성곱 계층의 구성이 동일하기 때문이다.
다만 **연산량은 스트라이드 쪽이 훨씬 적다.** 풀링 방식은 28×28 전체에 합성곱을 계산한 뒤 버리지만,
스트라이드 방식은 애초에 14×14 위치에서만 계산한다.

**어느 쪽이 나은가?** MNIST에서는 차이가 거의 없다. 다만 일반적으로는 각자의 자리가 있다.
최대 풀링은 '가장 강한 반응만 남긴다'는 성질 덕분에 작은 위치 변화에 더 둔감해서 분류 모델의 특징 추출기에 잘 맞는다.
반대로 스트라이드 합성곱은 축소 방식 자체를 학습하므로 더 유연하고, 특히 **정보를 최대한 보존해야 하는 경우**
(예: 11장에서 다룰 생성 모델)에 선호된다. 최근 모델에서는 스트라이드 합성곱을 쓰는 경우가 늘고 있다.

**주의할 점**은 두 합성곱 계층 사이에 **활성화 계층을 반드시 넣어야 한다**는 것이다.
풀링을 뺀 김에 ReLU까지 빼면 합성곱 두 개가 선형 변환 하나로 합쳐져 버려, 계층을 쌓은 의미가 사라진다.
3장에서 다룬 '활성화 함수 없는 다층 신경망은 단층과 같다'는 이야기가 합성곱에서도 그대로 적용된다.

### 문제 검토

- **적절성: 도전 문제로 적합.** 본문은 '합성곱 → 활성화 → 풀링'을 한 덩어리로 소개하는데,
  이 문제는 그 덩어리를 깨고 **풀링이 필수가 아니라는 것**을 알게 한다. 본문에서 배운 것을 조합해 풀 수 있는
  범위 안에 있으면서도 새로운 통찰을 준다. 좋은 도전 문제다.
- **[검토] 출력 형태를 명시한 것이 좋다.** `(B, 32, 7, 7)`이라는 목표가 주어져 있어 헤매지 않고,
  스트라이드 출력 크기 계산을 직접 해 보게 만든다. 제약("두 개의 합성곱 계층")도 답을 좁혀 준다.
- **[검토] 풀링 방식과 비교하라는 요구를 넣으면 더 좋겠다.** "결과를 확인해 보자"로 끝나면 정확도만 보고 만다.
  풀링 방식과 **파라미터 수, 연산량, 성능**을 견주어 보라고 하면 두 방식의 성격 차이가 드러난다.
  이 문제의 진짜 가치는 '되는구나'가 아니라 '무엇이 다른가'에 있다.
- **[검토] 스트라이드가 힌트로 필요한지.** 본문 p10~11에서 스트라이드를 이미 설명했으므로 힌트 없이도 풀 수 있다.
  도전 문제이니 그대로 두는 것이 적절하다.

**윤문안**

> **5-8** [도전 문제] 풀링 계층 없이 합성곱 계층만으로 `(B, 1, 28, 28)` 형태의 입력 텐서를 `(B, 32, 7, 7)` 형태로 줄인 후,
> 이를 분류기 계층으로 전달해 숫자를 분류하는 모델을 만들고, 결과를 확인해 보자.
> 단, 모델의 특징 추출기는 두 개의 합성곱 계층을 포함해야 한다.
> 완성한 모델을 풀링을 사용한 본문의 모델과 파라미터 수, 분류 성능 면에서 비교해 보자.